In [1]:
# importing libraries
import asyncio
import httpx 

In [3]:
# Semaphore handling maximum requests to prevent from waiting in queue.

# Setting configuration
CONCURRANCY_LIMIT = 3
QUEUE_TIMEOUT = 5.0

# Worker industry grade 
async def protected_ai_worker(client, url, sem, task_id):
    try:
        async with asyncio.timeout(QUEUE_TIMEOUT):
            async with sem:
                print(f"Slot Acquired for Task {task_id} entering critical zone")
                response = await client.get(url)
                print(f"Slot Released for Task {task_id} with status code: {response.status_code}")
                return (url, response.status_code)
    except TimeoutError:
        print(f"TIMEOUT: Task {task_id} dropped becuase it spent <{QUEUE_TIMEOUT}s")
        return {url , f"Error: time out for queuing request"}

    except Exception as err:
        print(f"FAILURE: Task {task_id} encountered unexpected erro: {err}")
        return {url, f"Error: {str(err)}"}

async def main():
    # instantiate global safe guard
    api_semaphore = asyncio.Semaphore(CONCURRANCY_LIMIT)
    target_apis = ['https://httpbin.org/get'] * 15

    async with httpx.AsyncClient() as client:
        tasks = (
            protected_ai_worker(client, url, api_semaphore, idx)
            for idx, url in enumerate(target_apis, start=1)
        )
        results = await asyncio.gather(*tasks, return_exceptions=True)

        print("Production pipeline execution summary")
        for url, status in results:
            print(f"Target: {url} -> Resolution: {status}")

await main()

Slot Acquired for Task 1 entering critical zone
Slot Acquired for Task 2 entering critical zone
Slot Acquired for Task 3 entering critical zone
Slot Released for Task 1 with status code: 200
Slot Acquired for Task 4 entering critical zone
Slot Released for Task 2 with status code: 200
Slot Acquired for Task 5 entering critical zone
Slot Released for Task 4 with status code: 200
Slot Released for Task 3 with status code: 200
Slot Acquired for Task 6 entering critical zone
Slot Acquired for Task 7 entering critical zone
Slot Released for Task 5 with status code: 200
Slot Acquired for Task 8 entering critical zone
Slot Released for Task 7 with status code: 200
Slot Released for Task 6 with status code: 200
Slot Acquired for Task 9 entering critical zone
Slot Acquired for Task 10 entering critical zone
Slot Released for Task 8 with status code: 200
Slot Acquired for Task 11 entering critical zone
Slot Released for Task 9 with status code: 200
Slot Released for Task 11 with status code: 200

In [11]:
# GLOBAL INITIALIZATION
CONCURRANCY_LIMIT = 3
API_TIMEOUT = 3.0
QUEUE_TIMEOUT = 10

# worker function
async def fetch_api_worker(client, url, sem, task_id):
    try:
        async with asyncio.timeout(QUEUE_TIMEOUT):
            async with sem:
                print(f"[SLOT ACQUIRED] Task {task_id} is entering network zone.")
                await asyncio.sleep(1.5)
                response = await client.get(url)
                print(f"[SLOT RELEASED] Task {task_id} is SUCCEDED {response.status_code}")
                return (url, response.status_code)
    except TimeoutError:
        print(f"TIMEOUT ERROR: Task {task_id} dropped becuase it spent {QUEUE_TIMEOUT}s")
        return {url, f"Error: Time out Error"}
    except Exception as err:
        print(f"FAILURE: Task {task_id} encountered unexpected error {err}")
        return {url, f"Error: {str(err)}"}

async def main():
    api_semaphore = asyncio.Semaphore(CONCURRANCY_LIMIT)
    target_url = ['https://httpbin.org/get']*10

    timeout = httpx.Timeout(API_TIMEOUT)
    # single client pool
    async with httpx.AsyncClient(timeout=timeout) as client:

        tasks = [
            fetch_api_worker(client, url, api_semaphore, idx)
            for idx, url in enumerate(target_url, start=1)

        ]

        results = await asyncio.gather(*tasks, return_exceptions=True)

        for result in results:
            if isinstance(result, BaseException):
                print(f"Request failed: {result}")
                continue
            url, status = result
            print(f"Target: {url} , Resolution: {status}")

await main()






    

[SLOT ACQUIRED] Task 1 is entering network zone.
[SLOT ACQUIRED] Task 2 is entering network zone.
[SLOT ACQUIRED] Task 3 is entering network zone.
FAILURE: Task 1 encountered unexpected error 
FAILURE: Task 2 encountered unexpected error 
FAILURE: Task 3 encountered unexpected error 
[SLOT ACQUIRED] Task 4 is entering network zone.
[SLOT ACQUIRED] Task 5 is entering network zone.
[SLOT ACQUIRED] Task 6 is entering network zone.
[SLOT RELEASED] Task 6 is SUCCEDED 200
[SLOT RELEASED] Task 4 is SUCCEDED 200
[SLOT RELEASED] Task 5 is SUCCEDED 200
[SLOT ACQUIRED] Task 7 is entering network zone.
[SLOT ACQUIRED] Task 8 is entering network zone.
[SLOT ACQUIRED] Task 9 is entering network zone.
[SLOT RELEASED] Task 9 is SUCCEDED 200
[SLOT RELEASED] Task 8 is SUCCEDED 200
[SLOT ACQUIRED] Task 10 is entering network zone.
[SLOT RELEASED] Task 7 is SUCCEDED 200
TIMEOUT ERROR: Task 10 dropped becuase it spent 10s
Target: Error:  , Resolution: https://httpbin.org/get
Target: Error:  , Resolution: h